# Imports

In [ ]:
import torch
print(f"Is CUDA available? {torch.cuda.is_available()}")
# Should return True

Is CUDA available? True


# Starter Code: Load The Data

In [2]:
import torch
from torchvision.transforms import v2
from PIL import Image

import os
import json
from typing import Callable

class VizWizLoader(torch.utils.data.Dataset):
    def __init__(self, strFolder: str, strAnnotationPath: str, fDataPercentage: float = 1.0,
                 tTransform: Callable[[torch.tensor], torch.tensor] = None) -> None:
        '''
        strFolder: path to unzip'd folder of VizWiz images
        strLabelPath: path to .json file containing the annotations
        fDataPercentage: percentage of available samples to use. Must be normalized between 0.0 and 1.0. Default: 1.0
        tTransform: optional place to connect PyTorch image transformations. Default: converts images to 3x224x224 tensors
        For the train and val splits, returns tuples of the form:
            (image, question text, binary label, answer texts)
        Otherwise, returns tuples of the form:
            (image, question text)
        '''
        self.strFolder = strFolder
        if self.strFolder[-1] != "/": self.strFolder += "/"
        self.tTransform = tTransform
        vecPaths = os.listdir(self.strFolder)
        self.strPrefix = vecPaths[0].split("_")[1]

        with open(strAnnotationPath, "r") as f:
            self.vecAnnos = json.load(f)

        self.iN = int(fDataPercentage * len(self.vecAnnos))

        if self.tTransform is None:
            self.tTransform = v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale = True), v2.RandomCrop(224)])
            self.tTransformUndersized = v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale = True), v2.Resize((224, 224))])

        return

    def __len__(self) -> int: return self.iN

    def __getitem__(self, idx: int) -> tuple:
        if idx > self.iN:
            print(f"Error! Tried to access index {idx} but only {self.iN} samples are available")
            return None

        strPath = self.strFolder + self.vecAnnos[idx]["image"]
        imX = Image.open(strPath)
        w, h = imX.size
        if w >= 224 and h >= 224: tX = self.tTransform(imX)
        else: tX = self.tTransformUndersized(imX)

        if self.strPrefix == "test":
            return tX, self.vecAnnos[idx]["question"]
        else:
            return tX, self.vecAnnos[idx]["question"], self.vecAnnos[idx]["answerable"], self.vecAnnos[idx]["answers"]

In [3]:
# 1. Path to your unzipped image folders
train_dir = r"C:\Users\Jalynn\CSCI5919_LabAssignment3\train\train"
val_dir = r"C:\Users\Jalynn\CSCI5919_LabAssignment3\val\val"
test_dir = r"C:\Users\Jalynn\CSCI5919_LabAssignment3\test\test"

# 2. Path to the .json annotation files (usually inside an Annotations folder)
train_anno = "annotations/train.json"
val_anno = "annotations/val.json"
test_anno = "annotations/test.json"

# --- TRAINING DATA (Enforce the 50% limit here!) ---
train_dataset = VizWizLoader(
    strFolder=train_dir,
    strAnnotationPath=train_anno,
    fDataPercentage=0.5  # This limits you to ~10,000 samples per the manual
)

# --- VALIDATION DATA ---
val_dataset = VizWizLoader(
    strFolder=val_dir,
    strAnnotationPath=val_anno,
    fDataPercentage=1.0
)

# --- TEST DATA (For your final predictions) ---
test_dataset = VizWizLoader(
    strFolder=test_dir,
    strAnnotationPath=test_anno,
    fDataPercentage=1.0
)

In [4]:
from torch.utils.data import DataLoader

batch_size = 32

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [5]:
# Pull one batch from the training loader
images, questions, labels, answers = next(iter(train_loader))

print(f"Image batch shape: {images.shape}")    # Should be [32, 3, 224, 224]
print(f"First question: {questions[0]}")      # The text question
print(f"First label: {labels[0]}")            # 0 or 1

Image batch shape: torch.Size([32, 3, 224, 224])
First question: What color is this top? 
First label: 1


# Text Tokenization
Building a vocab from 10,000 training samples, mapping each unique word to an integer

In [6]:
from collections import Counter
import re

def basic_tokenizer(text):
    # Lowercase and remove punctuation
    text = text.lower()
    text = re.sub(r'[^\w\s]', '', text)
    return text.split()

# 1. Build Vocabulary from your 10,000 samples
word_counts = Counter()
for i in range(len(train_dataset)):
    _, question, _, _ = train_dataset[i]
    word_counts.update(basic_tokenizer(question))

# 2. Create word-to-index mapping (reserve 0 for padding, 1 for unknown)
vocab = {word: i+2 for i, (word, _) in enumerate(word_counts.most_common(5000))}
vocab["<PAD>"] = 0
vocab["<UNK>"] = 1

def encode_text(text, max_len=20):
    tokens = basic_tokenizer(text)
    encoded = [vocab.get(token, 1) for token in tokens]
    # Pad or truncate
    if len(encoded) < max_len:
        encoded += [0] * (max_len - len(encoded))
    else:
        encoded = encoded[:max_len]
    return torch.tensor(encoded)

print(f"Vocab size: {len(vocab)}")
print(f"Example encoding: {encode_text('Is my Bose turned on?')}")

Vocab size: 2697
Example encoding: tensor([   4,   32, 2582,  276,   11,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0])


In [7]:
# That output is exactly what you want to see before feeding data into a Transformer. It tells us that your preprocessing pipeline is working correctly for Task 1.

# Here is the breakdown of what those numbers actually represent for your model:

# 1. Vocab Size: 2697
# This means that out of your 10,000 training samples, there are 2,697 unique words that appeared frequently enough to be indexed.

# Any word in the test set that wasn't in this list of 2,697 will be treated as an <UNK> (Unknown) token.

# This is a solid "goldilocks" number—large enough to understand most questions, but small enough that your RTX 3070 Ti won't run out of VRAM storing a massive embedding table.

# 2. The Tensor (Numerical Encoding)
# The tensor [4, 32, 2582, 276, 11, 0, 0, ...] is the "machine-readable" version of your question.

# The Non-Zero Numbers (4, 32, ...): These are the specific IDs for the words in "Is my Bose turned on?". For example, the number 4 might represent the word "is," and 32 might represent "my."

# The Zeros (0, 0, 0...): These are Padding Tokens (<PAD>). Since neural networks require every input in a batch to be the same size, we "pad" shorter questions with zeros to hit your max_len=20.

# 3. The Sequence Length (20)
# Your tensor has exactly 20 elements.

# Why this matters: In your report's Methods section, you should mention that you chose a fixed sequence length of 20. This is a design choice: if a user asks a very long question, it will be "truncated" (cut off), but for VizWiz, most questions are short, so 20 is a safe, efficient limit.

In [8]:
# make sure labels are ready for the loss function
# Should be torch.int64 with 0s and 1s
print(f"Label type: {labels.dtype}, Label values: {labels[:5]}")

Label type: torch.int64, Label values: tensor([1, 1, 1, 1, 1])


# Custom Multi-Modal Architecture
Satisfies requirement of self-attention and multi-modal inputs with a simple CNN for images and an Embedding + Transformer for text

In [14]:
import torch.nn as nn

import torch.nn as nn

class CustomVQAModel(nn.Module):
    # CHANGE 1: Added activation_type parameter
    def __init__(self, vocab_size, embed_dim=128, num_heads=4, activation_type="ReLU"):
        super(CustomVQAModel, self).__init__()
        
        # Helper to pick the activation
        if activation_type == "GELU":
            self.act = nn.GELU()
        elif activation_type == "LeakyReLU":
            self.act = nn.LeakyReLU()
        else:
            self.act = nn.ReLU() # Default
            
        # 1. Image Branch (Basic CNN)
        self.image_feat = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, stride=2, padding=1),
            self.act, # CHANGE 2: Using self.act instead of nn.ReLU()
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, kernel_size=3, stride=2, padding=1),
            self.act, # CHANGE 2
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten()
        )
        
        # 2. Text Branch (Self-Attention)
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        encoder_layer = nn.TransformerEncoderLayer(d_model=embed_dim, nhead=num_heads, batch_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=1)
        self.text_flatten = nn.Linear(embed_dim, 32)
        
        # 3. Fusion and Classifier
        self.classifier = nn.Sequential(
            nn.Linear(32 + 32, 64),
            self.act, # CHANGE 3: Using self.act
            nn.Linear(64, 1)
        )

    def forward(self, images, questions):
        img_v = self.image_feat(images)
        text_v = self.embedding(questions)
        text_v = self.transformer(text_v)
        text_v = text_v.mean(dim=1)
        text_v = self.text_flatten(text_v)
        combined = torch.cat((img_v, text_v), dim=1)
        logits = self.classifier(combined)
        return logits.squeeze()

# Instantiate for Challenge 1
# model_c1 = CustomVQAModel(len(vocab)).to("cuda")
print("Model initialized on GPU.")

Model initialized on GPU.


# Challenge #1

In [ ]:
import torch.optim as optim
from tqdm import tqdm

# List of activations to test
activations_to_test = ["ReLU", "GELU", "LeakyReLU"]
experiment_results = {}

for act_name in activations_to_test:
    print(f"\n>>> STARTING EXPERIMENT: {act_name} <<<")
    
    # 1. Initialize a FRESH model and optimizer for each trial
    # Using the 'activation_type' parameter we added to your class
    model_exp = CustomVQAModel(len(vocab), activation_type=act_name).to(device)
    optimizer = optim.Adam(model_exp.parameters(), lr=1e-4)
    criterion = nn.BCEWithLogitsLoss()
    
    # 2. Training Loop (5 Epochs)
    for epoch in range(5):
        model_exp.train()
        loop = tqdm(train_loader, total=len(train_loader), desc=f"{act_name} Epoch {epoch+1}")
        for batch in loop:
            imgs, questions_raw, labels, _ = batch
            questions_encoded = torch.stack([encode_text(q) for q in questions_raw]).to(device)
            imgs, labels = imgs.to(device), labels.to(device).float()
            
            optimizer.zero_grad()
            outputs = model_exp(imgs, questions_encoded)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            loop.set_postfix(loss=loss.item())

    # 3. Validation Section (Runs once after training each model)
    model_exp.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for batch in val_loader:
            imgs, questions_raw, labels, _ = batch
            questions_encoded = torch.stack([encode_text(q) for q in questions_raw]).to(device)
            imgs, labels = imgs.to(device), labels.to(device)
            
            outputs = model_exp(imgs, questions_encoded)
            predictions = (torch.sigmoid(outputs) > 0.5).int()
            correct += (predictions == labels).sum().item()
            total += labels.size(0)
    
    acc = 100 * correct / total
    experiment_results[act_name] = acc
    print(f"DONE: {act_name} achieved {acc:.2f}% Accuracy")

    # Clean up GPU memory before the next model starts
    del model_exp
    torch.cuda.empty_cache()

# 4. Final Comparison for your report
print("\n" + "="*30)
print("FINAL RESULTS TABLE")
print("="*30)
for name, accuracy in experiment_results.items():
    print(f"{name:<12} | Accuracy: {accuracy:.2f}%")


>>> STARTING EXPERIMENT: ReLU <<<


ReLU Epoch 5: 100%|██████████| 321/321 [03:05<00:00,  1.73it/s, loss=0.71] 


DONE: ReLU achieved 69.53% Accuracy

>>> STARTING EXPERIMENT: GELU <<<


GELU Epoch 1:  11%|█         | 36/321 [00:21<02:52,  1.65it/s, loss=0.534]

## Check Accuracy / Validation
Need to know how well the model does on the validation set to include in the report

# Export File for Challenge #1

In [ ]:
# Assuming GELU won the experiment
winner_act = "GELU"

In [ ]:
# 1. Re-initialize and Re-train (or just run this once if you already have the winner trained)
best_model = CustomVQAModel(len(vocab), activation_type=winner_act).to(device)
optimizer = optim.Adam(best_model.parameters(), lr=1e-4)
criterion = nn.BCEWithLogitsLoss()

# Quick 5-epoch train for the final submission
for epoch in range(5):
    best_model.train()
    for batch in train_loader:
        imgs, questions_raw, labels, _ = batch
        questions_encoded = torch.stack([encode_text(q) for q in questions_raw]).to(device)
        imgs, labels = imgs.to(device), labels.to(device).float()
        optimizer.zero_grad()
        outputs = best_model(imgs, questions_encoded)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

Saved 100 predictions to jalynn_nicoly_challenge1.pkl
Preview: tensor([1, 0, 1, 1, 1, 1, 1, 1, 1, 1])


In [ ]:
import torch

# 1. Set model to evaluation mode
best_model.eval()

# 2. Get the specific 100 samples (indices 100-199)
test_predictions = []

with torch.no_grad():
    for i in range(100, 200):
        img, question = test_dataset[i]
        
        # Add batch dimension and move to GPU
        img = img.unsqueeze(0).to(device)
        question_enc = encode_text(question).unsqueeze(0).to(device)
        
        # Forward pass
        output = best_model(img, question_enc)
        
        # Threshold at 0.5 for binary classification
        pred = (torch.sigmoid(output) > 0.5).int()
        test_predictions.append(pred.item())

# 3. Save with the exact required filename format
submission_tensor = torch.tensor(test_predictions)
filename = "jalynn_nicoly_challenge1.pkl"
torch.save(submission_tensor, filename)

print(f"Saved 100 predictions using {winner_act} to {filename}")
print(f"Preview: {submission_tensor[:10]}")